# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra-Jahangir/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** In the warehouse's `fact_content_daily_performance` table, one row = one content page, for one client, on one day. For my Lane 3 clustering work, I aggregate this up so that one row = one content page's totals across the whole month.

**Time window:** March 2026 (`month=2026-03`). This is a mid-panel month, not the `_sample` table — the `_sample` table is exactly the final month (June 2026), and using it now risks letting future data leak into logic meant to look only at the past.

I verify both of these claims with real queries in Section 3 below.

In [2]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Correct way to authenticate DuckDB with Hugging Face
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("Connection ready. Target file:", FACT_PATH)

Connection ready. Target file: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 2. Fields: feature / label / context / excluded

**Feature fields** (what I'll cluster on — all knowable from March data alone):
- `impressions`, `clicks` — raw search visibility signals, summed over the month
- `gsc_avg_position` — search ranking position, averaged over the month
- `sessions`, `engaged_sessions` — analytics activity, summed/ratioed over the month

**Label field:**
- None for real use — Lane 3 clustering has no ground-truth label. I only build a temporary fake target later, purely to demonstrate leakage — never used in my actual clustering.

**Context fields** (used for grouping/joining, not as clustering inputs):
- `content_hash_id`, `client_hash_id` — pseudonymized IDs, used only to join and group rows, never as numeric features
- `ga4_data_available` — a flag I filter on, not a feature itself

**Excluded fields:**
- Any raw text, URL, title, or query fields — excluded because they're either not shipped in this release or would risk exposing private information if they were
- No product-computed decision flags exist in this data to exclude, but I confirm I am not using any such field

In [3]:
# Confirm the columns I'm planning to use actually exist in the table
sample_cols = con.execute(f"""
    SELECT *
    FROM read_parquet('{FACT_PATH}')
    LIMIT 1
""").df()

print(sample_cols.columns.tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

Below: three verification queries (grain, scope, availability), a five-feature frame with "knowable at the decision moment" reasoning, and the deliberate leakage demonstration.

**Query 1 — grain check**

In [4]:
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{FACT_PATH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows breaking the expected grain (should be EMPTY):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows breaking the expected grain (should be EMPTY):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []


**Query 2 — scope check**

In [5]:
scope_check = con.execute(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{FACT_PATH}')
""").df()

print(scope_check)

   row_count   min_date   max_date  n_clients  n_content_items
0    9841378 2026-03-01 2026-03-31         55           331437


**Query 3 — availability check (`IS TRUE`)**

In [6]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{FACT_PATH}')
""").df()

print(availability_check)

survivors = con.execute(f"""
    SELECT COUNT(*) AS surviving_rows
    FROM read_parquet('{FACT_PATH}')
    WHERE ga4_data_available IS TRUE
""").df()

print(survivors)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0
   surviving_rows
0          413966


**Five features, each knowable at the decision moment:**
- `total_impressions_month` — only uses March rows, nothing from later
- `total_clicks_month` — same, only March data
- `avg_position_month` — average of daily positions, only from March
- `active_session_days` — count of days with real sessions, only from March
- `engagement_rate_month` — ratio built entirely from March numbers

In [9]:
features = con.execute(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions_month,
        SUM(gsc_clicks) AS total_clicks_month,
        AVG(gsc_avg_position) AS avg_position_month,
        COUNT(DISTINCT CASE WHEN ga4_sessions > 0 THEN report_date END) AS active_session_days,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate_month
    FROM read_parquet('{FACT_PATH}')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions_month,total_clicks_month,avg_position_month,active_session_days,engagement_rate_month
0,content_eb0aeedbcfaf2712,57.0,0.0,23.591837,5,0.000000
1,content_651b8ba180f9beff,83.0,1.0,15.804936,10,0.142857
2,content_1f39e904c7351258,103.0,0.0,18.672149,4,0.000000
3,content_5b619639be88dea7,140.0,4.0,6.491842,8,0.090909
4,content_3a3e193ec1e76e3b,89.0,1.0,12.381138,7,0.142857
5,content_a46986d3796592f9,673.0,4.0,9.597664,8,0.250000
6,content_1021a22410889b2a,294.0,2.0,23.775931,10,0.000000
7,content_232aa83d13706e4b,198.0,1.0,14.053422,8,0.111111
8,content_cc1ce98b4eb10e0c,16.0,0.0,66.714286,7,0.142857
9,content_f216815b76a1bdb8,27.0,0.0,23.472222,10,0.100000


**The trap:** I'll add one feature derived directly from a fake future-looking label, show the score jump toward suspiciously high, then delete it and keep the honest number.

In [10]:
NEXT_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

next_month = con.execute(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM read_parquet('{NEXT_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()

demo = features.merge(next_month, on="content_hash_id", how="inner")
demo["declined_next_month"] = (demo["impressions_april"] < demo["total_impressions_month"]).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_honest = demo[["total_impressions_month", "total_clicks_month", "avg_position_month",
                  "active_session_days", "engagement_rate_month"]].fillna(0)
y = demo["declined_next_month"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))
print("Honest accuracy (no leakage):", round(honest_score, 3))

Honest accuracy (no leakage): 0.946


In [12]:
demo["LEAKY_impressions_april"] = demo["impressions_april"]

X_leaky = demo[["total_impressions_month", "total_clicks_month", "avg_position_month",
                 "active_session_days", "engagement_rate_month", "LEAKY_impressions_april"]].fillna(0)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
leaky_score = accuracy_score(y_test2, model_leaky.predict(X_test2))
print("Leaky accuracy (with the answer smuggled in):", round(leaky_score, 3))

Leaky accuracy (with the answer smuggled in): 1.0


In [13]:
demo = demo.drop(columns=["LEAKY_impressions_april"])
print("Leaky feature removed. Honest score going forward:", round(honest_score, 3))

Leaky feature removed. Honest score going forward: 0.946


## 4. Data limits

This data can never prove causation — it can only show association, since nothing here is from a controlled experiment.

**Named limitation:** client history is unbalanced — some clients have far more tracked history than others, and rows before a client's GA4 tracking started have GA4 columns zero-filled with `ga4_data_available = FALSE` rather than truly reflecting zero engagement. My March slice may also lean toward the behavior of whichever clients have the most content in this particular month (see the real numbers below) rather than representing every client evenly.

In [14]:
client_spread = con.execute(f"""
    SELECT client_hash_id, COUNT(*) AS row_count
    FROM read_parquet('{FACT_PATH}')
    GROUP BY client_hash_id
    ORDER BY row_count DESC
    LIMIT 10
""").df()

print(client_spread)

            client_hash_id  row_count
0  client_625b6439094e23e4     988497
1  client_3ffa76342f366962     904847
2  client_73cda7b4e4f265ea     869640
3  client_08a6a72ff48e62c0     851275
4  client_62f4a7e64f5e0096     756660
5  client_65de48885f4ef01b     426307
6  client_23a62021009f63c4     423613
7  client_ba65e80a1116ae41     410409
8  client_2b4306c3ed003f01     375906
9  client_fef1a8f436438636     335379


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it

- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.